# gramlab — stage 3: more data for the 25M writer, fixed decoding, fresh retrieval set

Input: the **gramlab** notebook's stage-2 output (Version 2: shards, `small_wiki` at step 26,686 / ppl 46.1, raw Wikipedia text,
BM25, teacher embeddings). *Save & Run All*, GPU T4 x2, Internet on. Cost ≈ 6 GPU-hours (1 h data, 4 h training, 0.5 h rest).

Why: after ~4 passes over the 217M-token mix the writer's curve had flattened (ppl 46.5 → 46.1 over the last 100M tokens), so more
epochs buy nothing — this run adds **400k more English Wikipedia articles (~400M tokens)** and trains one pass over the new mix.
The visible answer quality problem of stage 2 was a decoding bug, not the model: the repetition penalty was applied to the retrieved
passage too, so copying "Tokyo" from the context was penalised into "Stockholm". Fixed in `generate()` (penalties now apply only to
generated tokens), and retrieval-conditioned chat decodes near-greedily (temperature 0.3).

In [ ]:
TRAIN_HOURS = 4.0
MORE_STEPS = 30000      # upper bound; the trainer fits the schedule to TRAIN_HOURS after 200 measured steps (~0.6 s/step -> ~23k steps)
EXTRA_EN_ARTICLES = 400000

In [ ]:
import os, subprocess, sys, time, glob, shutil, json
T0 = time.time()
def sh(cmd, must=True):
    print('\n$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if must and r.returncode != 0:
        raise SystemExit(f'command failed ({r.returncode}): {cmd}')
    return r.returncode
os.chdir('/kaggle/working')
if not os.path.exists('gramlab'):
    sh('git clone --depth 1 https://github.com/furevych/gramlab.git')
os.chdir('/kaggle/working/gramlab')
os.makedirs('external', exist_ok=True); os.makedirs('results', exist_ok=True); os.makedirs('data', exist_ok=True)
for url, d in [('https://github.com/gucorpling/amalgum', 'external/amalgum'),
               ('https://github.com/UniversalDependencies/UD_English-GUM', 'external/UD_English-GUM')]:
    if not os.path.exists(d):
        sh(f'GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 {url} {d}')
# the stage-2 output: prefer the one with the furthest-trained small_wiki checkpoint
import torch
cands = sorted(glob.glob('/kaggle/input/**/work/runs/small_wiki/ckpt.pt', recursive=True),
               key=lambda p: -torch.load(p, map_location='cpu', weights_only=False)['step'])
if not cands:
    raise SystemExit('Attach the gramlab notebook output (stage 2) as an input')
src = cands[0].split('/work/')[0]; print('previous output:', src, '| step', torch.load(cands[0], map_location='cpu', weights_only=False)['step'])
if not os.path.exists('work'):
    shutil.copytree(f'{src}/work', 'work')
    for f in glob.glob('work/teacher/passages.npy'):
        os.remove(f)                                 # tied to the old BM25 passage order; rebuilt only if needed later
for f in ['simplewiki.jsonl', 'enwiki.jsonl', 'wiki_qa.jsonl', 'stsb.jsonl']:
    if os.path.exists(f'{src}/data/{f}') and not os.path.exists(f'data/{f}'):
        shutil.copy(f'{src}/data/{f}', 'data/')
for f in glob.glob(f'{src}/results/*'):
    if not os.path.exists('results/' + os.path.basename(f)):
        shutil.copy(f, 'results/')
print('checkpoints:', sorted(glob.glob('work/runs/*/ckpt.pt')))

## 1. 400k more English Wikipedia articles (continuing after the first 120k), new shards

In [ ]:
if not os.path.exists('data/enwiki2.jsonl'):
    sh(f'python scripts/fetch_wikipedia.py --lang en --skip 120000 --max_articles {EXTRA_EN_ARTICLES} --out data/enwiki2.jsonl')
sh('python -m gramlab.prepare --config configs/small_wiki.json')
for c in ['amalgum', 'simplewiki', 'enwiki', 'enwiki2', 'gum_test']:
    m = json.load(open(f'work/bpe16000/shards/{c}/meta.json')); print(f"{c:11} {m['n_words']:>12,} words -> {m['n_tokens']:>12,} tokens")
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 2. Continue the writer on the new mix (one pass over ~620M tokens), schedule fitted to `TRAIN_HOURS`

In [ ]:
sh(f'python -m gramlab.train --config configs/small_wiki.json --resume --more_steps {MORE_STEPS} --set train.lr=2e-4 --set train.max_hours={TRAIN_HOURS}')
for line in open('work/runs/small_wiki/log.jsonl'):
    r = json.loads(line)
    if r['step'] % 2000 == 0 or r is None: print({k: r[k] for k in ('step', 'eval_ppl', 'eval_bits_per_byte', 'tokens_seen')})
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 3. BM25 over all three Wikipedia files, scoreboard, retrieval Q&A, fine-tunes

In [ ]:
WIKI = '--wiki data/simplewiki.jsonl --wiki data/enwiki.jsonl --wiki data/enwiki2.jsonl'
shutil.rmtree('work/bm25', ignore_errors=True)
sh(f'python -m gramlab.retrieval build {WIKI} --out work/bm25 --max_words 100')
sh(f'python -m gramlab.retrieval eval {WIKI} --index work/bm25 --n 3000 --out results/retrieval_bm25.json')
sh(f'python scripts/make_rag_qa.py {WIKI} --index work/bm25 --out data/rag_qa.jsonl --n 30000')
sh('python -m gramlab.finetune --init_from work/runs/small_wiki/ckpt.pt --corpus rag_qa --out small_wiki_rag --epochs 2')
sh('python -m gramlab.finetune --init_from work/runs/small_wiki/ckpt.pt --corpus wiki_qa --out small_wiki_sft --epochs 3', must=False)
sh('python -m gramlab.evaluate --ckpt work/runs/small_wiki/ckpt.pt --corpus gum_test --per_pos --out results/eval_small_wiki.json > results/eval_small_wiki.log', must=False)
print(f'elapsed {(time.time()-T0)/60:.0f} min')

## 4. Answers with retrieval (fixed decoding: penalties only on generated tokens, temperature 0.3) vs without

In [ ]:
import torch
sys.path.insert(0, '/kaggle/working/gramlab')
from gramlab.ckpt import load_checkpoint
from gramlab.tokenizer import USER, ASSISTANT, END, EOS
from gramlab.retrieval import BM25Index, format_rag_user
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
index = BM25Index.load('work/bm25')
models = {n: load_checkpoint(f'work/runs/{n}/ckpt.pt', dev) for n in ['small_wiki_rag', 'small_wiki_sft'] if os.path.exists(f'work/runs/{n}/ckpt.pt')}
def answer(name, user_text, temperature, top_k=40, top_p=0.9, rep=1.2, max_new=80):
    model, tok, state = models[name]
    ids = [USER] + tok.encode(user_text) + [END, ASSISTANT]
    x = torch.tensor([ids[-(model.ctx - 1):]], dtype=torch.long, device=dev)
    out = model.generate(x, max_new_tokens=max_new, temperature=temperature, top_k=top_k, top_p=top_p, repetition_penalty=rep, stop_ids=(END, EOS))
    return tok.decode(out[0, x.size(1):].tolist())
for q in ['Tell me about Belgrade.', 'Tell me about Paris.', 'Who was Napoleon?', 'What is the capital of Japan?', 'Tell me about the Danube.',
          'Who was Albert Einstein?', 'What is photosynthesis?', 'What is a volcano?', 'who is churchhill', 'Tell me about Zxqv Blorptown.']:
    passages = index.retrieve(q, k=1, max_words=90)
    print('Q:', q)
    for p in passages: print(f"   retrieved [{p['score']:.1f}] {p['title']}: {p['text'][:140]}...")
    if 'small_wiki_rag' in models: print('   with retrieval >', answer('small_wiki_rag', format_rag_user(passages or [{'title': '(no match)', 'text': 'No passage found.'}], q), 0.3))
    if 'small_wiki_sft' in models: print('   no retrieval   >', answer('small_wiki_sft', q, 0.7))
    print()

## 5. Trim

In [ ]:
for d in ['external', '.git']:
    shutil.rmtree(os.path.join('/kaggle/working/gramlab', d), ignore_errors=True)
os.system('du -sh /kaggle/working/gramlab/work/* /kaggle/working/gramlab/data /kaggle/working/gramlab/results 2>/dev/null')
print(f'total run time {(time.time()-T0)/3600:.2f} h')